In [1]:
!python -m pip install lightning

import numpy as np
import pandas as pd

import os

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.transforms import v2


import pytorch_lightning as pl
from pytorch_lightning import Trainer, callbacks
from pytorch_lightning.loggers import TensorBoardLogger
import datetime


print(os.getcwd())

C:\Users\flash\Desktop\Jobvorbereitungen\Projekte\DataScience\ImageClassification


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    accelerator = "gpu"
    workers = 1
    from google.colab import drive
    drive.mount('/content/drive')
    root_dir = "./drive/MyDrive/Colab Data/ResidualNeuralNetwork"
    torch.cuda.memory.empty_cache()
else:
    device = torch.device("cpu")
    accelerator = "cpu"
    workers = 1
    root_dir = "."

In [3]:
!set "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True"

In [4]:
# prepare the Data and DataAugmentation
batch_size = 300
height = 256
width = 256
classes = 2
shuffle = True
valid_size = 0.1

Image_augmentation_pipeline = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), # v2.ToTensor() is apparently depricatd and this new line is the replacement
    v2.RandomHorizontalFlip(p = 0.5),
    v2.RandomVerticalFlip(p= 0.5),
    v2.RandomRotation(30), # in degrees
    v2.ColorJitter(brightness = 0.3, contrast = 0.3, saturation = 0.3, hue = 0.3),
    v2.RandomGrayscale(p = 0.5),
    v2.RandomApply([v2.RandomCrop(size = 25)], p = 0.5),
    v2.Resize((width, height))
])


#### create train and validation dataloader
train_val_dataset = datasets.ImageFolder(f"{root_dir}/Data/Cats_Dogs/train", transform = Image_augmentation_pipeline)

num_train = len(train_val_dataset)
indices = list(range(num_train))
split = int(np.floor(valid_size * num_train))

if shuffle:
    np.random.seed(79)
    np.random.shuffle(indices)

train_idx, valid_idx = indices[split:], indices[:split]
train_sampler = torch.utils.data.sampler.SubsetRandomSampler(train_idx)
valid_sampler = torch.utils.data.sampler.SubsetRandomSampler(valid_idx)
train_loader = torch.utils.data.DataLoader(train_val_dataset, batch_size=batch_size, sampler=train_sampler, num_workers = workers)
valid_loader = torch.utils.data.DataLoader(train_val_dataset, batch_size=batch_size, sampler=valid_sampler, num_workers = workers)

# Currently the labels are indices range(0, classes). In the pytorch lightnign class, the labels will be one-hot encoded before training
# In Integers: 0 is Cat, 1 is Dog.
# In OneHot: [1, 0] is Cat, [0, 1] is Dog.

FileNotFoundError: [WinError 3] Das System kann den angegebenen Pfad nicht finden: './Data/Cats_Dogs/train'

In [ ]:
earlyStop = callbacks.EarlyStopping(monitor='val_loss', patience=3, min_delta = 1e-10)
# checkpPointTraining: Saves the weights per epoch to continue training in case of interruptions.
checkPointTraining = callbacks.ModelCheckpoint(dirpath = f"{root_dir}/Checkpoints", filename = "Pyboard_Test_{epoch}.weights.ckpt", monitor='val_loss', verbose=0, save_top_k = -1, save_weights_only=True)


tensorboard_callback = TensorBoardLogger(
    save_dir = f"{root_dir}/Checkpoints",
    name = "PyBoard_Checkpoint",
)

In [ ]:
checkPointTraining.state_key

In [ ]:
# Start TensorBoard server once trainign has started over the anaconda prompter
###### tensorboard --logdir=Checkpoints

# Open browser to http://localhost:6006
# View real-time loss curves, histograms, and model graphs

In [ ]:
# Build and Compile the Model
class ResidualBlock(nn.Module):
    """
    Class to innitiate a residual block
    """
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x) # in case that the new size is different from teh residual matrix, make the residual matrix fittin gto the new size
        out += residual
        out = self.relu(out)
        return out




class ResNet_Lightning(pl.LightningModule):
    def __init__(self, block, layers, num_classes = classes):
        """
        Build the network. the Variable Block wil be an uncalled instance of teh ResidualBlock class which gets called for the 4 make_layer arguments
        """
        super().__init__()
        self.num_classes = num_classes
        self.in_filter = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU()) # size after layer: 128 * 128 (becasue of stride = 2 and padding the size gets halved)
        self.max_pool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1) # size after layer: 64 * 64
        self.layer0 = self.make_layer(block, 64, layers[0], stride = 1) # size after layer: 64 * 64   afterwards self.in_filters = 64
        self.layer1 = self.make_layer(block, 128, layers[1], stride = 2) # sizea fter layer: 32 * 32     afterwards self.in_filters = 128
        # self.layer2 = self.make_layer(block, 256, layers[2], stride = 2) # size after layer: 16 * 16     afterwards self.in_filters = 256
        # self.layer3 = self.make_layer(block, 512, layers[3], stride = 2) # size after layer: 8 * 8     afterwards self.in_filters = 512
        self.avg_pool = nn.AvgPool2d(7, stride=1) # . Rsulting shape: (None, 512, 2, 2)
        self.flatten = nn.Flatten()
        # Before the Dense network, the conv output will be flattened using view(). 2 * 2 * 512 = 2048
        self.fc1 = nn.Linear(26*26*128, 1024) # for shape (None, 128, 32, 32) if done after self.layer1. (32 + 2*padding - kernel_size)/stride + 1 = 26
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, self.num_classes)
        self.relu = nn.ReLU()

    def configure_optimizers(self):
        # "weigth decay" is L2 Regularizer
        optimizer = torch.optim.Adam(self.parameters(), lr= 1e-3, weight_decay = 1e-10)
        # scheduler = StepLR(optimizer, gamma = self.hp.lr_decay_gamma, step_size = self.hp.lr_decay_stepsize)
        return [optimizer]


    def make_layer(self, block, filters, blocks, stride=1):
        """
        Implements the Residual Block into the ResNet.
        """
        downsample = None
        if stride != 1 or self.in_filter != filters:

            downsample = nn.Sequential(
                nn.Conv2d(self.in_filter, filters, kernel_size=1, stride=stride),
                nn.BatchNorm2d(filters),
            )
        layers = []
        layers.append(block(self.in_filter, filters, stride, downsample)) # size of layers/channels gets increased from the old self.in_filters to the new self.infilters.
        self.in_filter = filters
        for i in range(1, blocks):
            layers.append(block(self.in_filter, filters))

        return nn.Sequential(*layers)

    def training_step(self, batch, batch_idx):
        """
        Implement a single training period with loss function. Training loss is logged (by default to Tensorboard)
        """
        # training_step defines the train loop.
        # it is independent of forward
        x, y = batch
        # y_onehot = nn.functional.one_hot(y, num_classes = self.num_classes)

        x = self.conv1(x)
        x = self.max_pool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        # x = self.layer2(x)
        # x = self.layer3(x)

        x = self.avg_pool(x)
        x = self.flatten(x)#  x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        y_pred = self.fc3(x)


        loss = torch.nn.functional.cross_entropy(y_pred, y)
        # Logging to TensorBoard (if installed) by default
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Implement a single validation period with loss function. Validation loss is logged (by default to Tensorboard)
        """
        x, y = batch
        # y_onehot = nn.functional.one_hot(y, num_classes = self.num_classes)

        x = self.conv1(x)
        x = self.max_pool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        # x = self.layer2(x)
        # x = self.layer3(x)

        x = self.avg_pool(x)
        x = self.flatten(x)#  x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        y_pred = self.fc3(x)

        loss = torch.nn.functional.cross_entropy(y_pred, y)
        # Logging to TensorBoard (if installed) by default
        self.log("val_loss", loss)

    def testing_step(self, batch, batch_idx):
        """
        Implement a single validation period with loss function. Validation loss is logged (by default to Tensorboard)
        """
        x, y = batch
        # y_onehot = nn.functional.one_hot(y, num_classes = self.num_classes)

        x = self.conv1(x)
        x = self.max_pool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        # x = self.layer2(x)
        # x = self.layer3(x)

        x = self.avg_pool(x)
        x = self.flatten(x)#  x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        y_pred = self.fc3(x)

        loss = torch.nn.functional.cross_entropy(y_pred, y)
        # Logging to TensorBoard (if installed) by default
        self.log("test_loss", loss)


    def forward(self, x):
        """
        For prediction after training
        """

        x = self.conv1(x)
        x = self.max_pool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        #x = self.layer2(x)
        #x = self.layer3(x)

        x = self.avg_pool(x)
        x = self.flatten(x)#  x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        y_pred = np.argmax(self.fc3(x).detach().numpy())

        if y_pred == 0:
            label_pred = "Cat"
        else:
            label_pred = "Dog"


        return label_pred

    

In [ ]:
if not os.listdir(f"{root_dir}/Models/Pytorch"):
    # Only do all of this when the Finished Model has not yet been generated
    epochs = 30
    trainer = Trainer(callbacks=[earlyStop, checkPointTraining],
                      logger = tensorboard_callback,
                      max_epochs = epochs,
                      accelerator = accelerator,
                      devices = workers,
                      log_every_n_steps=1
                 )
    # Pyboard_Test_{epoch}.weights.ckpt
    SavePoints = [i for i in os.listdir(f"{root_dir}/Checkpoints") if i.startswith("Pyboard_Test") and i.endswith("weights.ckpt")]
    try:
        # Do checkpoints alerady exist ?
        initial_epoch = np.max([int(i.split(".")[0].split("_")[-1]) for i in SavePoints])
    except:
        initial_epoch = 0

    print(initial_epoch)

    # Either Start trainign from Scratch if no checkpoints exist or continue training from checkpoint
    ResNet = ResNet_Lightning(ResidualBlock, [3, 4, 6, 3]).to(device)
    if initial_epoch == 0:
        # load training weights
        trainer.fit(model = ResNet, train_dataloaders = train_loader, val_dataloaders = valid_loader)
    else:
        trainer.fit(model = ResNet, train_dataloaders = train_loader, val_dataloaders = valid_loader, ckpt_path=f"{root_dir}/Checkpoints/Pyboard_Test_{initial_epoch}.weights.ckpt")

In [ ]:
torch.save(ResNet, f"{root_dir}/Models/Pytorch/BestModel.save")
ResNet = torch.load(f"{root_dir}/Models/Pytorch/BestModel.save", weights_only = False)

In [ ]:
Image_augmentation_pipeline = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), # v2.ToTensor() is apparently depricatd and this new line is the replacement
    v2.Resize((width, height))
])

import glob
from PIL import Image

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, path, transform=None):
        self.image_paths = glob.glob(path + '*.jpg')
        self.transform = transform

    def __getitem__(self, index):
        x = Image.open(self.image_paths[index])
        if self.transform is not None:
            x = self.transform(x)

        return x

    def __len__(self):
        return len(self.image_paths)

test_dataset = TestDataset(f"{root_dir}/Data/Cats_Dogs/test/", transform = Image_augmentation_pipeline)
test_loader = torch.utils.data.DataLoader(test_dataset, num_workers = workers)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

for image in test_dataset:
    fig = plt.figure()
    ax = fig.add_subplot()
    # PyTorch uses (channels, width, height) for images, but matplotlib uses (width, height, channels). Use .permute to change it from PyTorch to Matplotlib format

    ax.imshow(image.permute(1, 2, 0))
    plt.show()

    label = ResNet(image.expand(1, -1, -1, -1))
    print(label)